# Part 4-A — RAG Architecture Design

This notebook describes and visualizes the RAG-based machine translation architecture.

**Goal:** Dynamic 5-shot example selection using vector similarity search, followed by in-context few-shot translation with Qwen 2.5 7B Instruct.

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import os

os.makedirs('../results', exist_ok=True)

## 4-A.1 Architecture Overview

The RAG pipeline has two phases:

### Phase 1: Offline Indexing (done once)
```
WMT16 Train Corpus
  (207K en-tr pairs)
        │
        ▼
Embedding Model                     FAISS Index
(multilingual-MiniLM)  ──embed──►  (IndexFlatIP)
        │                           207K × 384 dim
        └─────────────────────────► Saved to disk
```

### Phase 2: Online Translation (per sentence)
```
Input EN sentence
        │
        ▼
Embedding Model  ──embed──► query vector (384 dim)
        │
        ▼
FAISS Similarity Search  ──► Top-5 (en, tr) pairs
        │
        ▼
Prompt Builder
  [5 examples + source sentence]
        │
        ▼
Qwen 2.5 7B Instruct  ──► Turkish translation
```

## 4-A.2 System Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#f8f9fa')

def box(ax, x, y, w, h, text, color='#4A90D9', text_color='white', fontsize=10):
    rect = FancyBboxPatch((x, y), w, h,
                          boxstyle="round,pad=0.1",
                          facecolor=color, edgecolor='white',
                          linewidth=2, zorder=3)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, color=text_color, fontweight='bold',
            wrap=True, zorder=4)

def arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#333333',
                                lw=2.0, connectionstyle='arc3,rad=0'))

# ─── OFFLINE PHASE (left) ───
ax.text(2.5, 9.5, 'OFFLINE PHASE (Indexing)', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#555555')

box(ax, 0.3, 7.8, 4.4, 0.9, 'WMT16 Train Corpus\n(207K en-tr pairs)', color='#2ecc71', text_color='white')
box(ax, 0.3, 6.3, 4.4, 0.9, 'Multilingual Embedding Model\n(paraphrase-multilingual-MiniLM)', color='#3498db')
box(ax, 0.3, 4.8, 4.4, 0.9, 'EN Sentence Embeddings\n(207K × 384 dim, float32)', color='#9b59b6')
box(ax, 0.3, 3.3, 4.4, 0.9, 'FAISS IndexFlatIP\n(Cosine similarity, L2-normalized)', color='#e74c3c')
box(ax, 0.3, 1.8, 4.4, 0.9, 'Saved Index + Corpus\n(disk cache: .faiss + .pkl)', color='#95a5a6', text_color='#2c3e50')

arrow(ax, 2.5, 7.8, 2.5, 7.2)
arrow(ax, 2.5, 6.3, 2.5, 5.7)
arrow(ax, 2.5, 4.8, 2.5, 4.2)
arrow(ax, 2.5, 3.3, 2.5, 2.7)

# ─── ONLINE PHASE (right) ───
ax.text(10.5, 9.5, 'ONLINE PHASE (Per-sentence)', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#555555')

box(ax, 8.3, 7.8, 4.4, 0.9, 'Input EN Sentence', color='#2ecc71', text_color='white')
box(ax, 8.3, 6.3, 4.4, 0.9, 'Embedding Model\n(same as indexing)', color='#3498db')
box(ax, 8.3, 4.8, 4.4, 0.9, 'Query Vector\n(384 dim)', color='#9b59b6')
box(ax, 8.3, 3.3, 4.4, 0.9, 'FAISS Search\nTop-5 similar EN sentences', color='#e74c3c')
box(ax, 8.3, 1.8, 4.4, 0.9, 'Top-5 (EN, TR) Example Pairs', color='#e67e22')
box(ax, 8.3, 0.3, 4.4, 0.9, 'RAG Prompt → Qwen 2.5 7B\n→ Turkish Translation', color='#1abc9c')

arrow(ax, 10.5, 7.8, 10.5, 7.2)
arrow(ax, 10.5, 6.3, 10.5, 5.7)
arrow(ax, 10.5, 4.8, 10.5, 4.2)
arrow(ax, 10.5, 3.3, 10.5, 2.7)
arrow(ax, 10.5, 1.8, 10.5, 1.2)

# Connect index to online search
ax.annotate('', xy=(8.3, 3.75), xytext=(4.7, 3.75),
            arrowprops=dict(arrowstyle='->', color='#e74c3c',
                            lw=2.5, linestyle='dashed',
                            connectionstyle='arc3,rad=0'))
ax.text(6.5, 4.0, 'Load cached\nindex', ha='center', fontsize=9, color='#e74c3c')

# Separator line
ax.axvline(x=7.0, ymin=0, ymax=1, color='#cccccc', linewidth=1.5, linestyle='--', zorder=1)

plt.title('RAG Machine Translation Architecture — WMT16 EN→TR', 
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../results/rag_architecture.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/rag_architecture.png')

## 4-A.3 Component Descriptions

### Indexing Process

1. Load the full WMT16 train split (~207K sentence pairs).
2. Extract the English side of each pair.
3. Encode all English sentences with `paraphrase-multilingual-MiniLM-L12-v2` (batch_size=64).
4. L2-normalize all embeddings (enables cosine similarity via inner product).
5. Add all 207K vectors to a `faiss.IndexFlatIP` (exact search, 384-dimensional).
6. Persist the FAISS index and the full pair list to disk for reuse.

**Total indexing time (estimated):** ~15-30 minutes on CPU for 207K pairs.

### Retrieval Flow

For each test sentence:
1. Embed the English query sentence using the same embedding model.
2. Run `index.search(query_emb, k=6)` — retrieve 6 candidates (extra 1 in case query is in index).
3. Filter out exact matches, return top 5.
4. Return the (EN, TR) pairs for the top-5 indices.

**Retrieval time:** <1ms per query (FAISS exact search is O(n·d)).

### Prompt Construction Pipeline

```
"You are an expert English-to-Turkish translator. 
 Here are 5 similar translation examples for reference:

 Example 1:
 English: [retrieved_en_1]
 Turkish: [retrieved_tr_1]
 ...
 Example 5:
 English: [retrieved_en_5]
 Turkish: [retrieved_tr_5]

 Now translate the following sentence. Output only the translation.

 English: [query_sentence]
 Turkish:"
```

### Translation Generation Workflow

- The prompt is formatted with Qwen's chat template via `apply_chat_template`.
- Temperature=0.1 (near-greedy) for deterministic, faithful translations.
- `max_new_tokens=256` — sufficient for Turkish translations of typical news sentences.
- Output is the text after the last `Turkish:` marker.

### Embedding Model Choice

| Property | Value |
|---|---|
| Model | `paraphrase-multilingual-MiniLM-L12-v2` |
| Languages | 50+ (includes EN and TR) |
| Embedding dim | 384 |
| Parameters | 117M |
| Similarity | Cosine (L2-normalized inner product) |
| Rationale | Lightweight, multilingual, designed for semantic similarity tasks |

### Example Selection Strategy

We retrieve based on **semantic similarity of the English source sentence**. This means:
- Similar topic/domain → examples with relevant vocabulary
- Similar syntactic structure → examples demonstrating the same grammatical patterns
- Similar length → examples of appropriate complexity

This is superior to random few-shot selection because the model receives examples that are directly applicable to the structure and vocabulary of the current query.